## Import libraries


In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm.auto import tqdm

MAIN_DATA_DIR = "/beegfs/halder/GITHUB/RESEARCH/soil-amelioration-scenarios/data"
INTERIM_DATA_DIR = os.path.join(MAIN_DATA_DIR, "interim")
PROCESSED_DATA_DIR = os.path.join(MAIN_DATA_DIR, "processed")

## Read the datasets


In [2]:
# Read the soil coordinates
soil_coords = gpd.read_file(
    os.path.join(MAIN_DATA_DIR, "raw", "Site_Soil_BZE_WGS84.gpkg")
)
soil_coords = soil_coords[["PointID", "NUTS_ID", "NUTS_NAME", "STATE_NAME", "geometry"]]

print(soil_coords.shape)
soil_coords.head()

(3099, 5)


,PointID,NUTS_ID,NUTS_NAME,STATE_NAME,geometry
0,2,DEF07,Nordfriesland,Schleswig-Holstein,POINT (8.41161 54.85992)
1,3,DEF07,Nordfriesland,Schleswig-Holstein,POINT (8.69714 54.86438)
2,4,DEF07,Nordfriesland,Schleswig-Holstein,POINT (8.7653 54.86698)
3,5,DEF07,Nordfriesland,Schleswig-Holstein,POINT (8.95903 54.86392)
4,6,DEF07,Nordfriesland,Schleswig-Holstein,POINT (9.07846 54.87069)


## Algorithm to prepare phenology data


In [ ]:
def process_phenology(crop):
    phen_data_path = os.path.join(
        "/beegfs/halder/GITHUB/RESEARCH/DWD-Phenology/data/phenology_processed_merged_(DE)",
        f"phenology_({crop})_(DE).csv",
    )
    phen_coords_paths = "/beegfs/halder/GITHUB/RESEARCH/DWD-Phenology/data/phenology_processed_(DE)/phenology_stations.gpkg"

    phen_data = pd.read_csv(phen_data_path, parse_dates=["date"])
    phen_coords = gpd.read_file(phen_coords_paths)
    phen_coords = phen_coords.to_crs("EPSG:3035")

    harvest_next_year = ["winter_wheat", "winter_rapeseed"]

    # Sowing
    sowing_df = phen_data[phen_data["phase"] == "sowing"]
    sowing_df = sowing_df.rename(
        columns={"date": "sowing_date", "julian_day": "sowing_doy"}
    )
    sowing_df["harvest_year"] = (
        sowing_df["reference_year"] + 1
        if crop in harvest_next_year
        else sowing_df["reference_year"]
    )

    # Emergence
    emergence_df = phen_data[phen_data["phase"] == "emergence"]
    emergence_df = emergence_df.rename(
        columns={"date": "emergence_date", "julian_day": "emergence_doy"}
    )
    emergence_df["harvest_year"] = (
        emergence_df["reference_year"] + 1
        if crop in harvest_next_year
        else emergence_df["reference_year"]
    )

    # Flowering
    flowering_df = phen_data[phen_data["phase"] == "flowering"]
    flowering_df = flowering_df.rename(
        columns={
            "date": "flowering_date",
            "julian_day": "flowering_doy",
            "reference_year": "harvest_year",
        }
    )

    # Maturity
    maturity_df = phen_data[
        (phen_data["phase"] == "maturity") | (phen_data["phase"] == "harvest")
    ]
    maturity_df = maturity_df.rename(
        columns={
            "date": "maturity_date",
            "julian_day": "maturity_doy",
            "reference_year": "harvest_year",
        }
    )

    print(sowing_df.shape, emergence_df.shape, flowering_df.shape, maturity_df.shape)

    # Merge the phenology dataframes
    merged_df = sowing_df[
        ["stations_id", "harvest_year", "object", "sowing_date", "sowing_doy"]
    ].copy()
    merged_df = pd.merge(
        left=merged_df,
        right=emergence_df[
            ["stations_id", "harvest_year", "emergence_date", "emergence_doy"]
        ],
        on=["stations_id", "harvest_year"],
        how="inner",
    )

    if crop != "sugar_beet":
        merged_df = pd.merge(
            left=merged_df,
            right=flowering_df[
                ["stations_id", "harvest_year", "flowering_date", "flowering_doy"]
            ],
            on=["stations_id", "harvest_year"],
            how="inner",
        )

    merged_df = pd.merge(
        left=merged_df,
        right=maturity_df[
            ["stations_id", "harvest_year", "maturity_date", "maturity_doy"]
        ],
        on=["stations_id", "harvest_year"],
        how="inner",
    )

    # validatity checks

    if crop != "sugar_beet":
        merged_df["valid_sequence"] = (
            (merged_df["sowing_date"] < merged_df["emergence_date"])
            & (merged_df["emergence_date"] < merged_df["flowering_date"])
            & (merged_df["flowering_date"] < merged_df["maturity_date"])
        )

    else:
        merged_df["valid_sequence"] = (
            merged_df["sowing_date"] < merged_df["emergence_date"]
        ) & (merged_df["emergence_date"] < merged_df["maturity_date"])

    # Keep only the valid rows
    merged_df = merged_df[merged_df["valid_sequence"]]
    merged_df.rename(columns={"object": "crop"}, inplace=True)
    merged_df.reset_index(drop=True, inplace=True)

    # Plot the data
    if crop != "sugar_beet":
        columns = ["sowing_doy", "emergence_doy", "flowering_doy", "maturity_doy"]
    else:
        columns = ["sowing_doy", "emergence_doy", "maturity_doy"]

    # For plotting
    # for c in columns:
    #     sns.histplot(merged_df[c], label=c, bins=20)

    # plt.title(crop)
    # plt.legend(loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.1))
    # plt.show()

    return merged_df

## Find the nearest neighbour for all the soil points per year


In [7]:
crops = [
    "winter_wheat",
    "winter_rapeseed",
    "maize",
    "spring_barley",
    "sugar_beet",
    "potato",
]

phen_coords_paths = "/beegfs/halder/GITHUB/RESEARCH/DWD-Phenology/data/phenology_processed_(DE)/phenology_stations.gpkg"
phen_coords = gpd.read_file(phen_coords_paths)
phen_coords.to_crs(crs="EPSG:25832", inplace=True)
soil_coords.to_crs(crs="EPSG:25832", inplace=True)

for crop in crops:
    phenology_df = process_phenology(crop)

    final_phenology_df = pd.DataFrame()

    for year in range(1979, 2026):
        phen_df = phenology_df[phenology_df["harvest_year"] == year]

        if phen_df.empty:
            continue

        phen_df = gpd.GeoDataFrame(
            pd.merge(
                left=phen_df,
                right=phen_coords[["stations_id", "geometry"]],
                on="stations_id",
                how="inner",
            )
        )

        phen_df = gpd.sjoin_nearest(left_df=soil_coords, right_df=phen_df, how="left")
        phen_df.drop_duplicates(subset="PointID", inplace=True)
        phen_df.drop(columns=["index_right", "stations_id"], inplace=True)

        final_phenology_df = pd.concat(
            (final_phenology_df, phen_df), axis=0, ignore_index=True
        )
        final_phenology_df["harvest_year"] = final_phenology_df["harvest_year"].astype(
            int
        )

    # Plot a test year
    # plot_year = np.random.choice(final_phenology_df["harvest_year"].unique())
    # final_phenology_df[final_phenology_df["harvest_year"] == plot_year].plot(
    #     column="sowing_doy", markersize=5, legend=True
    # )
    # plt.title(f"Crop: {crop} | Year: {plot_year}")
    # plt.show()

    # Save the data
    out_path = os.path.join(
        PROCESSED_DATA_DIR, "phenology", "main", f"phenology_{crop}.csv"
    )
    final_phenology_df.drop(columns=["geometry"], inplace=True)
    final_phenology_df.to_csv(out_path, index=False)

(110078, 12) (101453, 12) (109858, 11) (132287, 11)
(62065, 12) (62955, 12) (74732, 11) (91462, 11)
(78229, 12) (75854, 12) (32007, 11) (72964, 11)
(87297, 12) (86926, 12) (77340, 11) (101519, 11)
(60693, 12) (57058, 12) (0, 11) (61740, 11)
(15455, 12) (15317, 12) (5856, 11) (8000, 11)


## Find the nearest neighbour for all the crop specific LAI points per year


In [44]:
crop_map_dict = {
    "Wheat": "winter_wheat",
    "Barley": "spring_barley",
    "Maize": "maize",
    "Potatoes": "potato",
    "Sugar Beet": "sugar_beet",
    "Rapeseed": "winter_rapeseed",
}

crop_samples_path = os.path.join(
    PROCESSED_DATA_DIR, "crop_type_samples", "crop_type_samples.gpkg"
)
crop_samples = gpd.read_file(crop_samples_path)
crop_samples["crop_type"] = crop_samples["crop_type"].replace(crop_map_dict)
crop_samples.to_crs(crs="EPSG:25832", inplace=True)

for crop in crops:

    if crop != "potato":
        phenology_df = process_phenology(crop)
    else:
        phenology_df = process_phenology(crop)
        # 1. Mean DOY per group
        mean_doy = (
            phenology_df.groupby(by=["stations_id", "crop"])[
                ["sowing_doy", "emergence_doy", "flowering_doy", "maturity_doy"]
            ]
            .median()
            .round(0)
            .astype(int)
            .reset_index()
        )

        future_years = pd.DataFrame(
            {"harvest_year": range(phenology_df["harvest_year"].max() + 1, 2023 + 1)}
        )

        future_df = mean_doy.merge(future_years, how="cross")

        def doy_to_date(year, doy):
            return pd.to_datetime(f"{year}-01-01") + pd.to_timedelta(doy - 1, unit="D")

        date_cols = ("sowing_date", "emergence_date", "flowering_date", "maturity_date")
        doy_cols = ("sowing_doy", "emergence_doy", "flowering_doy", "maturity_doy")

        year_start = pd.to_datetime(future_df["harvest_year"].astype(str) + "-01-01")

        for date_col, doy_col in zip(date_cols, doy_cols):
            future_df[date_col] = year_start + pd.to_timedelta(
                future_df[doy_col] - 1, unit="D"
            )

        future_df["valid_sequence"] = True
        future_df = future_df[
            [
                "stations_id",
                "harvest_year",
                "crop",
                "sowing_date",
                "sowing_doy",
                "emergence_date",
                "emergence_doy",
                "flowering_date",
                "flowering_doy",
                "maturity_date",
                "maturity_doy",
                "valid_sequence",
            ]
        ]
        phenology_df = future_df.copy()

    final_phenology_df = pd.DataFrame()

    for year in range(2017, 2022):
        phen_df = phenology_df[phenology_df["harvest_year"] == year]
        crop_samples_filtered = crop_samples[
            (crop_samples["crop_type"] == crop) & (crop_samples["year"] == year)
        ]

        if phen_df.empty:
            continue

        phen_df = gpd.GeoDataFrame(
            pd.merge(
                left=phen_df,
                right=phen_coords[["stations_id", "geometry"]],
                on="stations_id",
                how="inner",
            )
        )

        phen_df = gpd.sjoin_nearest(
            left_df=crop_samples_filtered, right_df=phen_df, how="left"
        )
        phen_df.drop_duplicates(subset="point_id", inplace=True)
        phen_df.drop(columns=["index_right", "stations_id"], inplace=True)

        final_phenology_df = pd.concat(
            (final_phenology_df, phen_df), axis=0, ignore_index=True
        )
        final_phenology_df["harvest_year"] = final_phenology_df["harvest_year"].astype(
            int
        )

    # Plot a test year
    # plot_year = np.random.choice(final_phenology_df["harvest_year"].unique())
    # final_phenology_df[final_phenology_df["harvest_year"] == plot_year].plot(
    #     column="sowing_doy", markersize=5, legend=True
    # )
    # plt.title(f"Crop: {crop} | Year: {plot_year}")
    # plt.show()

    # Save the data
    out_path = os.path.join(
        PROCESSED_DATA_DIR, "phenology", f"phenology_{crop}_LAI.csv"
    )
    final_phenology_df.drop(columns=["geometry"], inplace=True)
    # final_phenology_df.to_csv(out_path, index=False)

(107070, 11) (97658, 11) (104565, 10) (126947, 10)
(58368, 11) (58869, 11) (69422, 10) (84648, 10)
(73231, 11) (70679, 11) (27737, 10) (70421, 10)
(86144, 11) (86089, 11) (76198, 10) (100729, 10)
(60404, 11) (56774, 11) (0, 10) (61430, 10)
(94303, 11) (86889, 11) (75006, 10) (90923, 10)
